# Setup & Library Installation

In [ ]:
# CELL 1

!pip install -q sentence-transformers faiss-cpu nltk scikit-learn pandas numpy

import pandas as pd
import numpy as np
import re
import nltk
import faiss
import warnings

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

print("✅ Environment ready. All NLP & RAG libraries loaded successfully.")

# Load & Explore the Dataset

In [ ]:
# CELL 2

df = pd.read_csv("/content/ai_interviewer_questions_dataset_500_balanced (1).csv")

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

print("\nAvailable job roles:")
print(df['job_role'].value_counts())

print("\nDifficulty distribution:")
print(df['difficulty'].value_counts())

print("\nSample record:")
df.sample(1).T

# Text Preprocessing

In [ ]:
# CELL 3:

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Full NLP preprocessing pipeline:
    lowercase -> clean -> tokenize -> remove stopwords -> lemmatize
    """
    if not isinstance(text, str):
        return ""

    text = text.lower()

    text = re.sub(r'[^a-z\s]', ' ', text)

    tokens = word_tokenize(text)

    clean_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) > 1
    ]

    return " ".join(clean_tokens)

df['question_clean']      = df['question'].apply(preprocess_text)
df['ideal_answer_clean']  = df['ideal_answer'].apply(preprocess_text)
df['keywords_clean']      = df['keywords'].apply(preprocess_text)


df['retrieval_text'] = df['question_clean'] + " " + df['keywords_clean']

print("Example of preprocessing:")
print("-" * 60)
print("Original question :", df['question'].iloc[0])
print("Cleaned question  :", df['question_clean'].iloc[0])
print("-" * 60)
print("Original keywords :", df['keywords'].iloc[0])
print("Cleaned keywords  :", df['keywords_clean'].iloc[0])

df[['question', 'question_clean', 'retrieval_text']].head(3)

# Train/Test split (stratified by job_role)

In [ ]:
# CELL 4:

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['job_role']
)

print("Train size:", train_df.shape[0])
print("Test size :", test_df.shape[0])

# Baseline evaluation (BEFORE fine-tuning) on test_df only


In [ ]:
# CELL 5:
base_model = SentenceTransformer('all-MiniLM-L6-v2')

base_embeddings = base_model.encode(df['retrieval_text'].tolist(), convert_to_numpy=True)
faiss.normalize_L2(base_embeddings)

def evaluate_retriever_on(eval_df, model, all_embeddings, k=3):
    hits_at_k = 0
    reciprocal_ranks = []

    for idx, row in eval_df.iterrows():
        true_id = row['question_id']
        query_vec = model.encode([row['follow_up_questions'].split('|')[0]], convert_to_numpy=True)
        faiss.normalize_L2(query_vec)

        sims = cosine_similarity(query_vec, all_embeddings)[0]
        top_k_idx = np.argsort(sims)[::-1][:k]
        retrieved_ids = df.iloc[top_k_idx]['question_id'].tolist()

        if true_id in retrieved_ids:
            hits_at_k += 1
            reciprocal_ranks.append(1 / (retrieved_ids.index(true_id) + 1))
        else:
            reciprocal_ranks.append(0)

    precision = hits_at_k / len(eval_df)
    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
    return precision, mrr

print("=== BASELINE (pretrained, before fine-tuning) — test set only ===")
for k_val in [1, 3, 5]:
    p, m = evaluate_retriever_on(test_df, base_model, base_embeddings, k=k_val)
    print(f"k={k_val}  Precision@{k_val}: {p:.3f}   MRR: {m:.3f}")

# Fine-tune the embedding model on TRAIN split only

In [ ]:
# CELL 6:

from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader

base_model = SentenceTransformer('all-MiniLM-L6-v2')

train_examples = []
for _, row in train_df.iterrows():
    question = row['question']
    ideal_answer = row['ideal_answer']
    keywords_text = row['keywords'].replace('|', ' ')

    train_examples.append(InputExample(texts=[question, ideal_answer]))
    train_examples.append(InputExample(texts=[question, keywords_text]))

    fu_raw = row['follow_up_questions']
    if isinstance(fu_raw, str) and fu_raw.strip():
        for fu in fu_raw.split('|'):
            fu = fu.strip()
            if fu:
                train_examples.append(InputExample(texts=[question, fu]))

print("Total training pairs:", len(train_examples))

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)  # batch أكبر = negatives أقوى
train_loss = losses.MultipleNegativesRankingLoss(base_model)

base_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=int(len(train_dataloader) * 0.1),
    optimizer_params={'lr': 2e-5},
    show_progress_bar=True
)

base_model.save("/content/finetuned_interview_embedder")
print("✅ Fine-tuning done. Model saved.")

# Evaluation AFTER fine-tuning — same test_df, same query type

In [ ]:
# CELL 7:

finetuned_model = SentenceTransformer("/content/finetuned_interview_embedder")

ft_embeddings = finetuned_model.encode(df['retrieval_text'].tolist(), convert_to_numpy=True)
faiss.normalize_L2(ft_embeddings)

print("=== AFTER fine-tuning — test set only (follow_up_questions as query) ===")
for k_val in [1, 3, 5]:
    p, m = evaluate_retriever_on(test_df, finetuned_model, ft_embeddings, k=k_val)
    print(f"k={k_val}  Precision@{k_val}: {p:.3f}   MRR: {m:.3f}")

# Sanity check — evaluate fine-tuned model on TRAIN set too


In [ ]:
# CELL 8:
print("=== AFTER fine-tuning — TRAIN set (sanity check for overfitting) ===")
for k_val in [1, 3, 5]:
    p, m = evaluate_retriever_on(train_df, finetuned_model, ft_embeddings, k=k_val)
    print(f"k={k_val}  Precision@{k_val}: {p:.3f}   MRR: {m:.3f}")

# Compare semantic_score before vs after fine-tuning (on a sample answer)


In [ ]:
# CELL 9

pretrained_model = SentenceTransformer('all-MiniLM-L6-v2')

sample_question = df[df['question_id'] == 'TCH003'].iloc[0]
sample_answer = (
    "I always stay professional and calm when a student misbehaves. "
    "Instead of shouting or embarrassing them in front of the class, I "
    "speak to them quietly and treat them with respect."
)

def sbert_semantic_score(model, answer, ideal):
    v1 = model.encode([answer], convert_to_numpy=True)
    v2 = model.encode([ideal], convert_to_numpy=True)
    faiss.normalize_L2(v1); faiss.normalize_L2(v2)
    return float(cosine_similarity(v1, v2)[0][0]) * 100

before_score = sbert_semantic_score(pretrained_model, sample_answer, sample_question['ideal_answer'])
after_score = sbert_semantic_score(finetuned_model, sample_answer, sample_question['ideal_answer'])

print(f"Semantic score BEFORE fine-tuning: {before_score:.1f}")
print(f"Semantic score AFTER fine-tuning : {after_score:.1f}")

# Generating Sentence Embeddings (RAG Encoding Step)

In [ ]:
# CELL 10

embedding_model = SentenceTransformer("/content/finetuned_interview_embedder")

question_embeddings = embedding_model.encode(
    df['retrieval_text'].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

print("✅ Embeddings generated successfully.")
print("Shape of embeddings matrix:", question_embeddings.shape)
print("Each question is now represented as a vector of size:", question_embeddings.shape[1])

print("\nSample embedding (first question, first 10 dimensions):")
print(question_embeddings[0][:10])

# Building the Vector Store (FAISS Index) - RAG Retrieval Component

In [ ]:
# CELL 11:

faiss.normalize_L2(question_embeddings)

embedding_dim = question_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dim)

faiss_index.add(question_embeddings)

print(f"FAISS index built successfully.")
print(f"Total vectors stored in index: {faiss_index.ntotal}")
print(f"Vector dimensionality: {embedding_dim}")

test_query = "How do you handle a difficult team member?"
test_query_vec = embedding_model.encode([test_query], convert_to_numpy=True)
faiss.normalize_L2(test_query_vec)

scores, indices = faiss_index.search(test_query_vec, k=3)

print(f"\nTest Query: '{test_query}'")
print("Top 3 most similar questions in the knowledge base:\n")
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    print(f"{rank}. [{df.iloc[idx]['job_role']}] {df.iloc[idx]['question']}  (score: {score:.4f})")

# RAG Retriever Function

In [ ]:
# CELL 12:

def retrieve_questions(job_role, difficulty=None, query=None, top_k=1, exclude_ids=None):
    """
    Retrieve relevant interview question(s) from the knowledge base.

    Parameters
    ----------
    job_role : str        -> target job role (e.g. "Nurse")
    difficulty : str/None -> "Easy", "Medium", "Hard" or None (any)
    query : str/None      -> semantic query text to rank candidates
                              (if None, a generic role-based query is used)
    top_k : int           -> number of questions to return
    exclude_ids : list    -> question_ids to exclude (avoid repeats)

    Returns
    -------
    pandas.DataFrame with the top_k retrieved question rows
    """
    exclude_ids = exclude_ids or []

    candidates = df[df['job_role'] == job_role].copy()
    if difficulty:
        candidates = candidates[candidates['difficulty'] == difficulty]
    if exclude_ids:
        candidates = candidates[~candidates['question_id'].isin(exclude_ids)]

    if candidates.empty:
        return candidates

    # Step 2: Semantic ranking among the filtered candidates
    if query is None:
        query = f"important interview question for a {job_role}"

    query_vec = embedding_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)

    candidate_indices = candidates.index.to_numpy()
    candidate_embeddings = question_embeddings[candidate_indices]

    sims = cosine_similarity(query_vec, candidate_embeddings)[0]
    candidates = candidates.assign(retrieval_score=sims)
    candidates = candidates.sort_values('retrieval_score', ascending=False)

    return candidates.head(top_k)

# Test the retriever

result = retrieve_questions(job_role="Data Analyst", difficulty="Easy", top_k=1)
print("Retrieved question:")
print(result[['question_id', 'job_role', 'difficulty', 'question', 'retrieval_score']])

# Answer Evaluation Engine — Semantic Keyword Matching

In [ ]:
## CELL 13

from nltk.tokenize import sent_tokenize
from nltk.stem import PorterStemmer
nltk.download('punkt')

stemmer = PorterStemmer()

def get_stem_set(text):
    """Return the set of word stems for a piece of text."""
    clean = preprocess_text(text)
    return set(stemmer.stem(tok) for tok in clean.split())


def keyword_semantic_score(keyword, user_answer_raw, user_answer_clean,
                            exact_score=1.0, semantic_score_val=0.7,
                            semantic_threshold=0.40):
    keyword_stems = get_stem_set(keyword)
    answer_stems = get_stem_set(user_answer_raw)

    # (a) Exact match via STEM overlap
    if keyword_stems and keyword_stems.issubset(answer_stems):
        return exact_score, "exact"

    # (b) Semantic match: compare keyword to EACH sentence, keep the strongest
    sentences = sent_tokenize(user_answer_raw) or [user_answer_raw]
    kw_vec = embedding_model.encode([keyword], convert_to_numpy=True)
    sent_vecs = embedding_model.encode(sentences, convert_to_numpy=True)
    faiss.normalize_L2(kw_vec)
    faiss.normalize_L2(sent_vecs)

    sims = cosine_similarity(kw_vec, sent_vecs)[0]
    best_sim = float(np.max(sims))

    if best_sim >= semantic_threshold:
        return semantic_score_val, "semantic"

    return 0.0, "missing"


def evaluate_answer(user_answer, question_row):
    ideal_answer = question_row['ideal_answer']
    keywords = [k.strip().lower() for k in question_row['keywords'].split('|')]
    user_answer_clean = preprocess_text(user_answer)

    user_vec = embedding_model.encode([user_answer], convert_to_numpy=True)
    ideal_vec = embedding_model.encode([ideal_answer], convert_to_numpy=True)
    faiss.normalize_L2(user_vec)
    faiss.normalize_L2(ideal_vec)
    semantic_score = float(cosine_similarity(user_vec, ideal_vec)[0][0])

    matched_keywords, exact_keywords, missing_keywords = [], [], []
    per_keyword_scores = []

    for kw in keywords:
        score, match_type = keyword_semantic_score(kw, user_answer, user_answer_clean)
        per_keyword_scores.append(score)
        if match_type == "exact":
            matched_keywords.append(kw)
            exact_keywords.append(kw)
        elif match_type == "semantic":
            matched_keywords.append(kw)
        else:
            missing_keywords.append(kw)

    keyword_score = (sum(per_keyword_scores) / len(keywords)) if keywords else 0

    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform(
        [user_answer_clean, preprocess_text(ideal_answer)]
    )
    tfidf_score = float(cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0])

    final_score = (0.5 * semantic_score + 0.3 * keyword_score + 0.2 * tfidf_score) * 100
    final_score = round(final_score, 1)

    if final_score >= 75:
        label = "Excellent"
    elif final_score >= 55:
        label = "Good"
    elif final_score >= 35:
        label = "Needs Improvement"
    else:
        label = "Weak"

    return {
        "semantic_score": round(semantic_score * 100, 1),
        "keyword_score": round(keyword_score * 100, 1),
        "tfidf_score": round(tfidf_score * 100, 1),
        "final_score": final_score,
        "label": label,
        "matched_keywords": matched_keywords,
        "exact_keywords": exact_keywords,
        "missing_keywords": missing_keywords
    }


# Re-test the SAME demo example as before

demo_question = df[df['question_id'] == 'TCH003'].iloc[0]

demo_answer = (
    "I always stay professional and calm when a student misbehaves. "
    "Instead of shouting or embarrassing them in front of the class, I "
    "speak to them quietly and treat them with respect. "
    "I make sure my classroom rules are clear from day one so students "
    "know exactly what is expected of them."
)

print("Question:", demo_question['question'])
print("\nUser Answer:", demo_answer)

result = evaluate_answer(demo_answer, demo_question)

print("\n--- Evaluation Result (after stemming fix) ---")
for k, v in result.items():
    print(f"{k}: {v}")

# Sentiment & Confidence Analysis (VADER)

In [ ]:
# CELL 14:

from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()


def analyze_sentiment(text):
    """
    Analyze the emotional tone / confidence level of a candidate's answer
    using VADER (lexicon-based sentiment analysis).

    Returns
    -------
    dict with the compound score and a human-readable confidence label.
    """
    scores = sia.polarity_scores(text)
    compound = scores['compound']

    if compound >= 0.5:
        tone_label = "Confident / Positive"
    elif compound >= 0.05:
        tone_label = "Mildly Positive"
    elif compound > -0.05:
        tone_label = "Neutral"
    elif compound > -0.5:
        tone_label = "Mildly Hesitant / Negative"
    else:
        tone_label = "Hesitant / Negative"

    return {
        "compound": round(compound, 3),
        "positive": round(scores['pos'], 3),
        "neutral": round(scores['neu'], 3),
        "negative": round(scores['neg'], 3),
        "tone_label": tone_label
    }


# Test on the SAME demo example

demo_question = df[df['question_id'] == 'TCH003'].iloc[0]

demo_answer = (
    "I always stay professional and calm when a student misbehaves. "
    "Instead of shouting or embarrassing them in front of the class, I "
    "speak to them quietly and treat them with respect. "
    "I make sure my classroom rules are clear from day one so students "
    "know exactly what is expected of them."
)

sentiment_result = analyze_sentiment(demo_answer)

print("Question:", demo_question['question'])
print("\nUser Answer:", demo_answer)

print("\n--- Sentiment Analysis Result ---")
for k, v in sentiment_result.items():
    print(f"{k}: {v}")


# Compare with a clearly hesitant/negative answer for contrast

hesitant_answer = (
    "I don't really know what to do when a student misbehaves. "
    "I usually get confused and I'm not sure if I'm handling it right, "
    "it's honestly quite stressful for me."
)

print("\n" + "-" * 60)
print("Contrast example (hesitant answer):", hesitant_answer)
print(analyze_sentiment(hesitant_answer))

# RAG-based Feedback Generation + Sentiment/Tone

In [ ]:
# CELL 15

def generate_feedback(evaluation_result, question_row, user_answer):
    criteria_list = [c.strip() for c in question_row['evaluation_criteria'].split('|')]
    missing_kw = evaluation_result['missing_keywords']
    matched_kw = evaluation_result['matched_keywords']
    exact_kw = evaluation_result['exact_keywords']
    semantic_only_kw = [kw for kw in matched_kw if kw not in exact_kw]

    score = evaluation_result['final_score']
    label = evaluation_result['label']

    feedback_parts = []
    feedback_parts.append(f"Overall assessment: {label} ({score}/100).")

    if exact_kw:
        feedback_parts.append(
            f"Strengths: you correctly used precise professional terms such as "
            f"{', '.join(exact_kw[:4])}."
        )

    if semantic_only_kw:
        feedback_parts.append(
            f"Good conceptual understanding: your answer conveyed the idea behind "
            f"{', '.join(semantic_only_kw)}, though you didn't use the exact term. "
            f"Using the precise professional vocabulary next time would strengthen your answer."
        )

    if missing_kw:
        feedback_parts.append(
            f"To improve: your answer didn't cover {', '.join(missing_kw)}, "
            f"which are important aspects of a complete answer to this question."
        )

    feedback_parts.append(
        f"This question is typically evaluated on: {'; '.join(criteria_list)}."
    )

    # Tone / Confidence feedback (only meaningful for
    # Behavioral / Communication style questions)

    tone_relevant_categories = ["Behavioral", "Communication"]
    if question_row['category'] in tone_relevant_categories:
        sentiment_result = analyze_sentiment(user_answer)
        tone_label = sentiment_result['tone_label']

        if tone_label in ["Confident / Positive", "Mildly Positive"]:
            feedback_parts.append(
                f"Tone: Your answer came across as {tone_label.lower()}, "
                f"which is a good sign for a behavioral question — interviewers "
                f"value candidates who sound composed and confident."
            )
        elif tone_label == "Neutral":
            feedback_parts.append(
                "Tone: Your answer was fairly neutral in tone. Adding a bit more "
                "conviction in how you describe your actions can make a stronger impression."
            )
        else:
            feedback_parts.append(
                f"Tone: Your answer came across as {tone_label.lower()}. Try rephrasing "
                f"with more confident, action-oriented language (e.g., 'I handled...' "
                f"instead of 'I'm not sure...')."
            )

    if score < 35:
        feedback_parts.append(
            "Tip: Review the core concept from scratch and structure your "
            "answer with a clear definition followed by an example."
        )
    elif score < 55:
        feedback_parts.append(
            "Tip: Your foundation is there, but add more specific details "
            "or a real-world example to strengthen your answer."
        )
    elif score < 75:
        feedback_parts.append(
            "Tip: Solid answer! Adding the missing points above would make "
            "it excellent."
        )
    else:
        feedback_parts.append(
            "Tip: Excellent, well-rounded answer. Keep this level of detail "
            "and structure in your real interview."
        )

    return "\n".join(feedback_parts)


# Test on the SAME demo

demo_question = df[df['question_id'] == 'TCH003'].iloc[0]

demo_answer = (
    "I always stay professional and calm when a student misbehaves. "
    "Instead of shouting or embarrassing them in front of the class, I "
    "speak to them quietly and treat them with respect. "
    "I make sure my classroom rules are clear from day one so students "
    "know exactly what is expected of them."
)

eval_res = evaluate_answer(demo_answer, demo_question)
feedback = generate_feedback(eval_res, demo_question, demo_answer)

print("Question:", demo_question['question'])
print("\nUser Answer:", demo_answer)

print("\n--- Evaluation ---")
for k, v in eval_res.items():
    print(f"{k}: {v}")

print("\n--- Generated Feedback ---")
print(feedback)

# Voice Recording (Colab Mic) + Speech-to-Text (Whisper)

In [ ]:
# CELL 16:


!pip install -q openai-whisper

import whisper
from google.colab import output
from base64 import b64decode
import time

RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));

var recorder, stream;

async function startRecording() {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  recorder = new MediaRecorder(stream);
  recorder.chunks = [];
  recorder.ondataavailable = e => recorder.chunks.push(e.data);
  recorder.start();
}

async function stopRecording() {
  recorder.stop();
  stream.getTracks().forEach(t => t.stop());
  await sleep(300);
  const blob = new Blob(recorder.chunks, { type: 'audio/webm' });
  const arrayBuffer = await blob.arrayBuffer();
  const bytes = new Uint8Array(arrayBuffer);
  let binary = '';
  for (let i = 0; i < bytes.length; i++) binary += String.fromCharCode(bytes[i]);
  return btoa(binary);
}
"""


def record_answer(seconds=15):
    """
    Records audio from the browser microphone for a fixed duration
    and saves it as a .webm file. Returns the file path.
    """

    output.eval_js(RECORD_JS + "\nstartRecording();", ignore_result=True)

    print(f"🎙️  Recording... speak now ({seconds} seconds)")
    time.sleep(seconds)

    audio_b64 = output.eval_js(RECORD_JS + "\nstopRecording();")
    audio_path = "user_answer.webm"

    with open(audio_path, "wb") as f:
        f.write(b64decode(audio_b64))

    print("✅ Recording finished.")
    return audio_path


print("Loading Whisper model (this runs once)...")
whisper_model = whisper.load_model("base")
print("✅ Whisper model loaded.")


def transcribe_audio(audio_path):
    """
    Transcribes a recorded audio file into text using Whisper.
    Returns the transcribed text (English).
    """
    result = whisper_model.transcribe(audio_path, language="en")
    return result["text"].strip()

# Interview Session Simulator (Text or Voice)

In [ ]:
# CELL 17:

def get_user_answer():
    """
    Lets the candidate choose how to answer: typed text or recorded voice.
    Returns the final answer as plain text (transcribed if voice).
    """
    while True:
        mode = input("\nHow would you like to answer? Type 't' for text or 'v' for voice: ").strip().lower()
        if mode in ['t', 'v']:
            break
        print("Please enter 't' for text or 'v' for voice.")

    if mode == 't':
        answer = input("\nYour answer: ")
    else:
        audio_path = record_answer(seconds=15)
        print("Transcribing your answer...")
        answer = transcribe_audio(audio_path)
        print(f"\n📝 Transcribed answer: {answer}")

    return answer


def run_interview_session(job_role):
    difficulty_levels = ["Easy", "Medium", "Hard"]
    session_results = []
    used_ids = []

    print("=" * 70)
    print(f" AI INTERVIEW COACH — Role: {job_role}")
    print("=" * 70)

    for i, difficulty in enumerate(difficulty_levels, start=1):
        retrieved = retrieve_questions(
            job_role=job_role,
            difficulty=difficulty,
            top_k=1,
            exclude_ids=used_ids
        )

        if retrieved.empty:
            print(f"\n(No question found for difficulty={difficulty}, skipping.)")
            continue

        question_row = retrieved.iloc[0]
        used_ids.append(question_row['question_id'])

        print(f"\n--- Question {i}/3 ({difficulty}) ---")
        print(question_row['question'])

        user_answer = get_user_answer()

        eval_result = evaluate_answer(user_answer, question_row)
        feedback = generate_feedback(eval_result, question_row, user_answer)

        print(f"\nScore: {eval_result['final_score']}/100 ({eval_result['label']})")
        print("\nFeedback:")
        print(feedback)

        session_results.append({
            "question_id": question_row['question_id'],
            "question": question_row['question'],
            "difficulty": difficulty,
            "category": question_row['category'],
            "user_answer": user_answer,
            "semantic_score": eval_result['semantic_score'],
            "keyword_score": eval_result['keyword_score'],
            "tfidf_score": eval_result['tfidf_score'],
            "final_score": eval_result['final_score'],
            "label": eval_result['label'],
            "feedback": feedback
        })

    print("\n" + "=" * 70)
    print("Interview session complete! Run Cell 12 for your final report.")
    print("=" * 70)

    return session_results


available_roles = df['job_role'].unique().tolist()

print("Welcome to the AI Interview Coach!")
print("Available job roles:")
for idx, role in enumerate(available_roles, start=1):
    print(f"  {idx}. {role}")

choice = input("\nSelect a job role (type the number): ")

try:
    selected_role = available_roles[int(choice) - 1]
except (ValueError, IndexError):
    print("Invalid choice, defaulting to 'Software Engineer'.")
    selected_role = "Software Engineer"

session_results = run_interview_session(job_role=selected_role)

# Final Interview Report + Visualization

In [ ]:
# CELL 18

import matplotlib.pyplot as plt
import textwrap

def _wrap(text, width=45):
    """Shorten long question text so the table stays readable."""
    return textwrap.shorten(text, width=width, placeholder="...")


def generate_final_report(session_results):
    if not session_results:
        print("No session results to report on.")
        return None

    report_df = pd.DataFrame(session_results)

    overall_avg = report_df['final_score'].mean()
    avg_semantic = report_df['semantic_score'].mean()
    avg_keyword = report_df['keyword_score'].mean()
    best_q = report_df.loc[report_df['final_score'].idxmax()]
    worst_q = report_df.loc[report_df['final_score'].idxmin()]

    # SECTION 1: Header

    print("╔" + "═" * 68 + "╗")
    print("║{:^68}║".format("FINAL INTERVIEW REPORT"))
    print("╚" + "═" * 68 + "╝")
    print(f"\n Overall Average Score : {overall_avg:5.1f} / 100")
    print(f" Avg. Semantic Score   : {avg_semantic:5.1f} / 100")
    print(f" Avg. Keyword Score    : {avg_keyword:5.1f} / 100")

    # SECTION 2: Per-question breakdown (clean table)

    print("\n" + "-" * 70)
    print(" PER-QUESTION BREAKDOWN")
    print("-" * 70)

    table_df = report_df.copy()
    table_df['question'] = table_df['question'].apply(_wrap)
    table_df = table_df.rename(columns={
        'difficulty': 'Difficulty',
        'question': 'Question',
        'final_score': 'Score',
        'label': 'Result'
    })

    print(table_df[['Difficulty', 'Question', 'Score', 'Result']]
          .to_string(index=False, justify='left'))

    # SECTION 3: Best / Weakest answers

    print("\n" + "-" * 70)
    print(" HIGHLIGHTS")
    print("-" * 70)
    print(f" 🟢 Strongest Answer  [{best_q['difficulty']}]")
    print(f"    \"{_wrap(best_q['question'], 60)}\"")
    print(f"    Score: {best_q['final_score']}/100")

    print(f"\n 🔴 Weakest Answer    [{worst_q['difficulty']}]")
    print(f"    \"{_wrap(worst_q['question'], 60)}\"")
    print(f"    Score: {worst_q['final_score']}/100")

    # SECTION 4: Coaching summary

    print("\n" + "-" * 70)
    print(" COACHING SUMMARY")
    print("-" * 70)

    if overall_avg >= 75:
        print(" Excellent performance overall! Your answers were well-structured,\n"
              " accurate, and covered the key concepts expected for this role.")
    elif overall_avg >= 55:
        print(" Good overall performance. Your answers show solid understanding,\n"
              " but adding more precise terminology and complete coverage of key\n"
              " concepts (see missing keywords above) would push you to excellent.")
    elif overall_avg >= 35:
        print(" Your performance needs improvement. While you understand the general\n"
              " ideas, your answers were missing several important concepts and\n"
              " specific terminology expected for this role.")
    else:
        print(" Your answers need significant work. Focus on reviewing the core\n"
              " concepts for this role from scratch before your next practice session.")

    if avg_semantic > avg_keyword + 15:
        print("\n 💡 Insight: Your conceptual/semantic understanding is stronger than your\n"
              "    use of precise professional terminology — try to name concepts explicitly.")
    elif avg_keyword > avg_semantic + 15:
        print("\n 💡 Insight: You use the right terminology, but try to explain concepts\n"
              "    more thoroughly with examples and reasoning, not just keywords.")

    print("\n" + "=" * 70 + "\n")

    # SECTION 5: Visualization

    fig, ax = plt.subplots(figsize=(8, 5))

    colors = ['#4CAF50' if s >= 75 else '#2196F3' if s >= 55 else '#FF9800' if s >= 35 else '#F44336'
              for s in report_df['final_score']]
    bars = ax.bar(report_df['difficulty'], report_df['final_score'], color=colors, width=0.5)

    ax.axhline(y=overall_avg, color='gray', linestyle='--', linewidth=1.2,
               label=f'Average ({overall_avg:.1f})')

    ax.set_ylim(0, 100)
    ax.set_ylabel("Score (0-100)", fontsize=11)
    ax.set_title("Interview Performance Summary", fontsize=13, fontweight='bold', pad=15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(frameon=False)

    for bar, score in zip(bars, report_df['final_score']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f"{score}", ha='center', fontweight='bold', fontsize=10)

    plt.tight_layout()
    plt.show()

    return report_df

# Generate the final report for the session run in Cell 9

final_report_df = generate_final_report(session_results)

# RAG Retriever Evaluation — Precision@k & MRR

In [ ]:
# CELL 19:

def evaluate_retriever(k=3):
    """
    Evaluates the FAISS-based retriever using a self-retrieval test:
    for every question in the knowledge base, we use its own text as
    a query and check whether the retriever ranks the correct
    question_id near the top of the results.

    Metrics computed
    -----------------
    Precision@k : fraction of queries where the correct question_id
                  appears anywhere in the top-k retrieved results.
    MRR         : Mean Reciprocal Rank — rewards ranking the correct
                  result HIGHER (1st place = 1.0, 2nd place = 0.5, ...).
    """
    hits_at_k = 0
    reciprocal_ranks = []

    for idx, row in df.iterrows():
        true_id = row['question_id']
        query_text = row['question']

        query_vec = embedding_model.encode([query_text], convert_to_numpy=True)
        faiss.normalize_L2(query_vec)

        scores, indices = faiss_index.search(query_vec, k=k)
        retrieved_ids = df.iloc[indices[0]]['question_id'].tolist()

        if true_id in retrieved_ids:
            hits_at_k += 1

        if true_id in retrieved_ids:
            rank = retrieved_ids.index(true_id) + 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)

    precision_at_k = hits_at_k / len(df)
    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)

    return precision_at_k, mrr

# Run the evaluation for a couple of k values

print("=" * 60)
print(" RAG RETRIEVER EVALUATION (Information Retrieval Metrics)")
print("=" * 60)

for k_val in [1, 3, 5]:
    precision, mrr = evaluate_retriever(k=k_val)
    print(f"\nk = {k_val}")
    print(f"  Precision@{k_val} : {precision:.3f}  ({precision*100:.1f}% of queries found the correct question in top-{k_val})")
    print(f"  MRR            : {mrr:.3f}  (1.0 = always ranked #1, closer to 0 = ranked low or missed)")

# BERTScore — Standard NLP Evaluation Metric

In [ ]:
# CELL 20

!pip install -q bert-score

from bert_score import score as bert_score_fn

def compute_bertscore(user_answer, ideal_answer):
    """
    Computes BERTScore between the user's answer and the ideal answer.

    Unlike our SBERT-based semantic_score (which compares the two
    texts as single sentence vectors), BERTScore compares them at the
    TOKEN level: every word in the user's answer is matched against
    its most similar word in the ideal answer using contextual BERT
    embeddings, then the matches are aggregated into Precision,
    Recall, and F1.

    Returns
    -------
    dict with precision, recall, and f1 (all in the 0-1 range).
    """
    P, R, F1 = bert_score_fn(
        [user_answer],
        [ideal_answer],
        lang="en",
        verbose=False
    )

    return {
        "bertscore_precision": round(P.item(), 3),
        "bertscore_recall": round(R.item(), 3),
        "bertscore_f1": round(F1.item(), 3)
    }


demo_question = df[df['question_id'] == 'TCH003'].iloc[0]

demo_answer = (
    "I always stay professional and calm when a student misbehaves. "
    "Instead of shouting or embarrassing them in front of the class, I "
    "speak to them quietly and treat them with respect. "
    "I make sure my classroom rules are clear from day one so students "
    "know exactly what is expected of them."
)

bert_result = compute_bertscore(demo_answer, demo_question['ideal_answer'])

print("Question:", demo_question['question'])
print("\nUser Answer:", demo_answer)

print("\n--- BERTScore Result ---")
for k, v in bert_result.items():
    print(f"{k}: {v}")

print("\n--- Compare with our existing SBERT-based semantic_score ---")
existing_eval = evaluate_answer(demo_answer, demo_question)
print(f"Our semantic_score (SBERT, sentence-level): {existing_eval['semantic_score']}")
print(f"BERTScore F1 (BERT, token-level):            {bert_result['bertscore_f1'] * 100:.1f}")

# Interactive Web Interface (Gradio)

In [ ]:
# CELL 21
!pip install -q gradio

import gradio as gr

DIFFICULTY_LEVELS = ["Easy", "Medium", "Hard"]
available_roles = df['job_role'].unique().tolist()

# Helper: fetch a question and format it for display

def _get_question_display(job_role, difficulty_idx, used_ids):
    retrieved = retrieve_questions(
        job_role=job_role,
        difficulty=DIFFICULTY_LEVELS[difficulty_idx],
        top_k=1,
        exclude_ids=used_ids
    )
    if retrieved.empty:
        return None, "⚠️ No more questions available for this role."

    q_row = retrieved.iloc[0]
    display_text = (
        f"**Question {difficulty_idx + 1}/3 "
        f"({DIFFICULTY_LEVELS[difficulty_idx]})**\n\n{q_row['question']}"
    )
    return q_row, display_text

# Step 1: Start Interview

def start_interview(job_role):
    state = {
        "job_role": job_role,
        "used_ids": [],
        "diff_idx": 0,
        "session_results": [],
        "current_question_id": None
    }

    q_row, question_display = _get_question_display(job_role, 0, [])
    if q_row is None:
        return state, question_display, "", ""

    state["current_question_id"] = q_row['question_id']
    return state, question_display, "", ""

# Step 2: Submit Answer (text OR voice)

def submit_answer(state, text_answer, audio_path):
    if not state or state.get("current_question_id") is None:
        return state, "⚠️ Please click 'Start Interview' first.", "", None, None

    # --- Get the answer: voice takes priority if recorded ---
    if audio_path:
        user_answer = transcribe_audio(audio_path)
    elif text_answer and text_answer.strip():
        user_answer = text_answer.strip()
    else:
        return state, "⚠️ Please type an answer or record your voice.", "", None, None

    question_row = df[df['question_id'] == state["current_question_id"]].iloc[0]

    # --- Evaluate using the SAME pipeline as the console version ---
    eval_result = evaluate_answer(user_answer, question_row)
    feedback = generate_feedback(eval_result, question_row, user_answer)

    state["used_ids"].append(question_row['question_id'])
    state["session_results"].append({
        "question_id": question_row['question_id'],
        "question": question_row['question'],
        "difficulty": DIFFICULTY_LEVELS[state["diff_idx"]],
        "category": question_row['category'],
        "user_answer": user_answer,
        "semantic_score": eval_result['semantic_score'],
        "keyword_score": eval_result['keyword_score'],
        "tfidf_score": eval_result['tfidf_score'],
        "final_score": eval_result['final_score'],
        "label": eval_result['label'],
        "feedback": feedback
    })

    score_display = f"### Score: {eval_result['final_score']}/100 ({eval_result['label']})\n\n{feedback}"

    state["diff_idx"] += 1

    if state["diff_idx"] < 3:
        q_row, next_question_display = _get_question_display(
            state["job_role"], state["diff_idx"], state["used_ids"]
        )
        state["current_question_id"] = q_row['question_id'] if q_row is not None else None
    else:
        next_question_display = "✅ **Interview complete!** Click 'Get Final Report' below."
        state["current_question_id"] = None

    return state, next_question_display, score_display, None, ""

# Step 3: Final Report (text summary + chart)

def show_final_report(state):
    if not state or not state.get("session_results"):
        return "No session data yet — complete an interview first.", None

    report_df = pd.DataFrame(state["session_results"])
    overall_avg = report_df['final_score'].mean()
    avg_semantic = report_df['semantic_score'].mean()
    avg_keyword = report_df['keyword_score'].mean()
    best_q = report_df.loc[report_df['final_score'].idxmax()]
    worst_q = report_df.loc[report_df['final_score'].idxmin()]

    summary_lines = [
        f"## 📊 Final Interview Report — {state['job_role']}",
        f"**Overall Average Score:** {overall_avg:.1f}/100",
        f"**Avg. Semantic Score:** {avg_semantic:.1f}/100  |  **Avg. Keyword Score:** {avg_keyword:.1f}/100",
        "",
        f"🟢 **Strongest:** [{best_q['difficulty']}] {best_q['question']} — {best_q['final_score']}/100",
        f"🔴 **Weakest:** [{worst_q['difficulty']}] {worst_q['question']} — {worst_q['final_score']}/100",
        ""
    ]

    if overall_avg >= 75:
        summary_lines.append("Excellent performance overall! Well-structured and accurate answers.")
    elif overall_avg >= 55:
        summary_lines.append("Good performance — more precise terminology would push you to excellent.")
    elif overall_avg >= 35:
        summary_lines.append("Needs improvement — several key concepts were missing.")
    else:
        summary_lines.append("Significant work needed — review core concepts for this role.")

    summary_text = "\n\n".join(summary_lines)

    # --- Build the chart as a matplotlib figure (returned, not shown) ---
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = ['#4CAF50' if s >= 75 else '#2196F3' if s >= 55 else '#FF9800' if s >= 35 else '#F44336'
              for s in report_df['final_score']]
    bars = ax.bar(report_df['difficulty'], report_df['final_score'], color=colors, width=0.5)
    ax.axhline(y=overall_avg, color='gray', linestyle='--', linewidth=1.2, label=f'Average ({overall_avg:.1f})')
    ax.set_ylim(0, 100)
    ax.set_ylabel("Score (0-100)")
    ax.set_title("Interview Performance Summary", fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(frameon=False)
    for bar, score in zip(bars, report_df['final_score']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f"{score}", ha='center', fontweight='bold')
    plt.tight_layout()

    return summary_text, fig


# Build the Gradio Interface

with gr.Blocks(title="AI Interview Coach") as demo:
    gr.Markdown(
        "# 🎯 AI Interview Coach\n"
        "An NLP & RAG-powered mock interview system. Choose your role, "
        "answer by **text or voice**, and get instant, grounded feedback."
    )

    session_state = gr.State({})

    with gr.Row():
        role_dropdown = gr.Dropdown(choices=available_roles, label="Select your job role", value=available_roles[0])
        start_btn = gr.Button("🚀 Start Interview", variant="primary")

    question_display = gr.Markdown("Click **Start Interview** to begin.")

    with gr.Tab("✍️ Text Answer"):
        text_input = gr.Textbox(label="Type your answer", lines=4, placeholder="Type your answer here...")
        submit_text_btn = gr.Button("Submit Text Answer")

    with gr.Tab("🎙️ Voice Answer"):
        audio_input = gr.Audio(sources=["microphone"], type="filepath", label="Record your answer")
        submit_audio_btn = gr.Button("Submit Voice Answer")

    feedback_display = gr.Markdown("")

    gr.Markdown("---")
    report_btn = gr.Button("📊 Get Final Report", variant="secondary")
    report_display = gr.Markdown("")
    report_chart = gr.Plot()

    # --- Wire up events ---
    start_btn.click(
        fn=start_interview,
        inputs=[role_dropdown],
        outputs=[session_state, question_display, feedback_display, text_input]
    )

    submit_text_btn.click(
        fn=submit_answer,
        inputs=[session_state, text_input, gr.State(None)],
        outputs=[session_state, question_display, feedback_display, audio_input, text_input]
    )

    submit_audio_btn.click(
        fn=submit_answer,
        inputs=[session_state, gr.State(""), audio_input],
        outputs=[session_state, question_display, feedback_display, audio_input, text_input]
    )

    report_btn.click(
        fn=show_final_report,
        inputs=[session_state],
        outputs=[report_display, report_chart]
    )

demo.launch(debug=True)